# 02 · QC and quantify (local, kallisto)
Per run: `.sra` → FASTQ → **FastQC** → *(optional fastp trim)* → **kallisto** (GENCODE vM25 transcripts) → delete FASTQ.
Peak extra disk ≈ 7 GB per run. kallisto needs only a few GB RAM, so no cluster is required; STAR (index ≈ 30 GB RAM) is the part for Euler.

Smart-seq2 libraries are unstranded, so kallisto defaults apply. **Decide on trimming only after reading the QC** (cell 4).

In [ ]:
import sys; sys.path.insert(0, '../scripts')
import helpers as h
runs = h.load_runs()

## 1 · Reference and index (~2 min, once)

In [ ]:
h.download_reference()
h.build_index()

index exists


## 2 · QC on all samples (all reads)
Quality differs between samples, so every downloaded run is checked on **all** reads: convert to a temporary FASTQ → FastQC → delete (≈ 3–6 min and ≈ 7 GB of temporary disk per run; interrupt-safe, finished runs are skipped). Re-run after more downloads finish. Tutorial focus: *per-base quality*, *adapter content*, *overrepresented sequences*, *duplication*; GC/base-content warnings are normal for RNA-seq.

In [ ]:
done = [s for s in runs.index if h.has_sra(s)]
for srr in done:
    h.fastqc(srr)
print('report:', h.multiqc())   # open this HTML: all samples side by side
h.fastqc_summary(done)

  SRR21355392 fastqc: 85s
  SRR21355388 fastqc: 82s
  SRR21355401 fastqc: 76s
  SRR21355396 fastqc: 122s
  SRR21355390 fastqc: 138s
  SRR21355386 fastqc: 47s
  SRR21355398 fastqc: 35s
  SRR21355394 fastqc: 38s
  SRR21355389 fastqc: 78s
  SRR21355385 fastqc: 71s
  SRR21355397 fastqc: 66s
  SRR21355393 fastqc: 50s
  SRR23914577 fastqc: 126s
  SRR23914574 fastqc: 106s
  SRR23914572 fastqc: 66s
  SRR23914571 fastqc: 77s
  SRR23914455 fastqc: 130s
  SRR23914452 fastqc: 77s
  SRR23914450 fastqc: 104s
  SRR23914454 fastqc: 110s
  SRR23914453 fastqc: 68s
  SRR23914451 fastqc: 84s
  SRR23914579 fastqc: 73s
  SRR23914578 fastqc: 53s
  SRR23914576 fastqc: 64s
  SRR23914575 fastqc: 120s
  SRR23914573 fastqc: 61s
report: /Users/jianzhouyao/Desktop/RNAseq_aging/results/qc/multiqc_raw.html


,Per base sequence quality,Per sequence quality scores,Adapter Content,Overrepresented sequences,Sequence Duplication Levels
SRR21355392_1,PASS,PASS,FAIL,WARN,WARN
SRR21355392_2,PASS,PASS,FAIL,WARN,WARN
SRR21355388_1,PASS,PASS,FAIL,WARN,WARN
SRR21355388_2,PASS,PASS,FAIL,WARN,WARN
SRR21355401_1,PASS,PASS,FAIL,WARN,WARN
SRR21355401_2,PASS,PASS,FAIL,WARN,WARN
SRR21355396_1,PASS,PASS,FAIL,WARN,WARN
SRR21355396_2,PASS,PASS,FAIL,WARN,WARN
SRR21355390_1,PASS,PASS,FAIL,WARN,WARN
SRR21355390_2,PASS,PASS,FAIL,WARN,WARN


## 3 · Trim or not?
Trim if **any** sample shows poor per-base quality at the read ends (median < 20) or adapter-content FAIL in the table / MultiQC report. kallisto tolerates mild issues, so uniform PASS means no trimming. To preview the effect on one run: `h.trim('SRR21355398')` → `results/qc/SRR21355398.fastp.html`.

In [ ]:
TRIM = True   # FastQC on all 27 runs: Nextera adapter FAIL (17-34 % of reads) + poly-G in every sample; quality PASS

## 4 · Test quantification on the one run

In [ ]:
h.process('SRR21355398', trim_reads=TRIM, keep_fastq=True)
h.qc_summary(runs)

,pairs_M,aligned_%,group
SRR21355392,17.0,62.1,aging 3mo
SRR21355388,17.7,62.9,aging 3mo
SRR21355401,15.6,60.6,aging 3mo
SRR21355396,27.1,62.3,aging 3mo
SRR21355398,6.8,62.6,aging 18mo


Expect pseudoaligned ≈ 60–80 % (mouse mRNA). Much lower → check the reference or strandedness before scaling up.

## 5 · All downloaded runs
Skips runs already quantified or not yet downloaded; interrupt-safe. Re-run when more downloads finish.

In [ ]:
print('TRIM =', TRIM)   # must be True: re-run the TRIM cell if the kernel was restarted
h.check_consistent(runs, TRIM)
for srr in runs.index:
    if h.has_sra(srr):
        h.process(srr, trim_reads=TRIM)
h.clean_tmp()
h.check_consistent(runs, TRIM)

## 6 · Summary

In [ ]:
h.qc_summary(runs)

,pairs_M,aligned_%,q30_%,dup_%,group
SRR21355392,17.0,62.1,NaN,NaN,aging 3mo
SRR21355388,17.7,62.9,NaN,NaN,aging 3mo
SRR21355401,15.6,60.6,NaN,NaN,aging 3mo
SRR21355396,27.1,62.3,NaN,NaN,aging 3mo
SRR21355390,28.2,61.9,88.7,8.0,aging 18mo
SRR21355386,9.3,65.5,86.1,3.3,aging 18mo
SRR21355398,6.8,62.6,NaN,NaN,aging 18mo
SRR21355394,7.8,65.3,86.2,2.9,aging 18mo
SRR21355389,16.9,63.1,88.1,7.2,aging 21mo
SRR21355385,15.5,63.5,85.6,4.3,aging 21mo


In [ ]:
h.status(runs)

,group,sra_GB,quantified
SRR,,,
SRR21355392,aging 3mo,1.88,True
SRR21355388,aging 3mo,1.94,True
SRR21355401,aging 3mo,1.67,True
SRR21355396,aging 3mo,3.03,True
SRR21355390,aging 18mo,3.23,True
SRR21355386,aging 18mo,1.13,True
SRR21355398,aging 18mo,0.78,True
SRR21355394,aging 18mo,0.95,True
SRR21355389,aging 21mo,1.96,True
